# Blob 4:Files
Extracts clinically/research-valuable **non-text files** (PDF, TIFF, JPEG, PNG) that the text
pipeline discards, decodes them, and saves them as real files in a UC Volume with a manifest table.

**Runs after `Blob 3:Merge`** in the same job (raw bytes still fresh in the prod landing zone).
Serverless. Self-scoping by `RUN_ID` via `mill_blob_extractor_history`.

Flow: RUN_ID → history-scoped distinct keys → join bronze (full grain) → in-scope content-type →
dedup to file key → retry-aware anti-join → exact raw re-read (EVENT_ID+ADC_UPDT) → **hash verify** →
size gate → bounded decode → temp write → structural validate → atomic rename → retry-aware manifest upsert.

Decoder: `%run ./blob_decoder` (byte-identical decode contract with Blob 2, preflight-free).

In [0]:
%pip install ocflzw-decompress pypdfium2 Pillow
dbutils.library.restartPython()

In [0]:
%run ./blob_decoder

In [0]:
# ==================== CONFIG (widgets) ====================
dbutils.widgets.text("RUN_ID", "", "RUN_ID (required for SOURCE=forward)")
dbutils.widgets.text("SOURCE", "dev_backfill", "forward | dev_backfill")
dbutils.widgets.text("SOURCE_TABLE", "8_dev.default.blob4_stage", "raw chunk source (pure raw mirror)")
dbutils.widgets.text("BRONZE_TABLE", "4_prod.bronze.mill_blob_text", "decoded-text bronze (authoritative content_type/raw_sha256)")
dbutils.widgets.text("HISTORY_TABLE", "4_prod.logs.mill_blob_extractor_history", "per-run ledger")
dbutils.widgets.text("CLINEVENT_TABLE", "4_prod.raw.mill_clinical_event", "EVENT_ID->PERSON_ID map")
dbutils.widgets.text("MANIFEST_TABLE", "8_dev.default.mill_blob_files", "output manifest")
dbutils.widgets.text("TARGET_VOLUME", "/Volumes/8_dev/default/dev_test_volume/files", "output volume root")
dbutils.widgets.text("TRUST", "Barts", "Trust filter")
dbutils.widgets.text("FILE_MAX_SIZE", str(10 * 1024 * 1024), "max decoded file bytes (skip above)")
dbutils.widgets.text("CREATE_MANIFEST_IF_MISSING", "true", "create manifest table if absent (dev)")

RUN_ID          = dbutils.widgets.get("RUN_ID").strip()
SOURCE          = dbutils.widgets.get("SOURCE").strip()
SOURCE_TABLE    = dbutils.widgets.get("SOURCE_TABLE").strip()
BRONZE_TABLE    = dbutils.widgets.get("BRONZE_TABLE").strip()
HISTORY_TABLE   = dbutils.widgets.get("HISTORY_TABLE").strip()
CLINEVENT_TABLE = dbutils.widgets.get("CLINEVENT_TABLE").strip()
MANIFEST_TABLE  = dbutils.widgets.get("MANIFEST_TABLE").strip()
TARGET_VOLUME   = dbutils.widgets.get("TARGET_VOLUME").rstrip("/")
TRUST           = dbutils.widgets.get("TRUST").strip()
FILE_MAX_SIZE   = int(dbutils.widgets.get("FILE_MAX_SIZE"))
CREATE_MANIFEST = dbutils.widgets.get("CREATE_MANIFEST_IF_MISSING").strip().lower() == "true"

IN_SCOPE_CT = ["application/pdf", "image/tiff", "image/jpeg", "image/png"]
TERMINAL_STATUSES = ["Extracted", "Skipped - oversize", "Skipped - not in-scope type"]
# Retryable (re-enter next run): Decode failed, Validation failed, Raw version mismatch, Raw missing.

import os, hashlib, traceback
from pyspark.sql import Window
from pyspark.sql import functions as F

print(f"SOURCE={SOURCE} RUN_ID={RUN_ID or '(none)'} SOURCE_TABLE={SOURCE_TABLE}")
print(f"MANIFEST={MANIFEST_TABLE} VOLUME={TARGET_VOLUME} FILE_MAX_SIZE={FILE_MAX_SIZE}")
if SOURCE == "forward" and not RUN_ID:
    raise ValueError("SOURCE=forward requires a RUN_ID (never scope by watermark).")

In [0]:
# ==================== MANIFEST TABLE (ensure exists) ====================
MANIFEST_DDL = f"""
CREATE TABLE IF NOT EXISTS {MANIFEST_TABLE} (
    EVENT_ID BIGINT, PERSON_ID BIGINT, ENCNTR_ID BIGINT, Trust STRING, ADC_UPDT TIMESTAMP,
    content_type STRING, file_ext STRING, byte_size BIGINT, blob_length_expected BIGINT,
    raw_sha256 STRING, file_sha256 STRING, volume_path STRING, decompressor_version INT,
    decompression_strategy STRING, STATUS STRING, attempt_count INT, last_error STRING,
    run_id STRING, source STRING, extracted_ts TIMESTAMP, extracted_date DATE
) USING DELTA PARTITIONED BY (file_ext)
"""
if CREATE_MANIFEST:
    spark.sql(MANIFEST_DDL)

def table_exists(name: str) -> bool:
    try:
        spark.table(name).schema
        return True
    except Exception:
        return False

In [0]:
# ==================== CANDIDATE SELECTION ====================
# `cand`: one row per FILE KEY (EVENT_ID, raw_sha256) with the chosen ADC_UPDT + content_type +
# BINARY_SIZE + ENCNTR_ID + Trust. Metadata ALWAYS comes from authoritative bronze (magic-decided
# content_type + Blob-2 raw_sha256). Retry-aware: excludes only terminal manifest keys.

bronze_inscope = (spark.table(BRONZE_TABLE)
    .where(F.col("Trust") == TRUST)
    .where(F.col("CONTENT_TYPE").isin(IN_SCOPE_CT))
    .where(F.col("raw_sha256").isNotNull())
    .select("EVENT_ID", "ADC_UPDT", "raw_sha256", "CONTENT_TYPE", "BINARY_SIZE",
            "ENCNTR_ID", "Trust", "VALID_FROM_DT_TM"))

if SOURCE == "forward":
    # Scope to exactly the keys this run produced (durable per-run ledger).
    hist = (spark.table(HISTORY_TABLE)
            .where(F.col("run_id") == RUN_ID)
            .select("EVENT_ID", "ADC_UPDT", "raw_sha256").distinct())
    cand0 = hist.join(bronze_inscope, ["EVENT_ID", "ADC_UPDT", "raw_sha256"], "inner")
else:
    # dev_backfill: staging is a PURE RAW MIRROR; scope bronze by the staged EVENT_IDs so metadata
    # stays authoritative. (Raw chunks come from SOURCE_TABLE in the re-read step.)
    stage_ids = spark.table(SOURCE_TABLE).select("EVENT_ID").distinct()
    cand0 = bronze_inscope.join(F.broadcast(stage_ids), ["EVENT_ID"], "inner")

# Deterministic dedup to ONE row per file key (bronze has up to 5x dups on EVENT_ID+raw_sha256).
w_key = Window.partitionBy("EVENT_ID", "raw_sha256").orderBy(
    F.col("ADC_UPDT").desc_nulls_last(),
    F.col("VALID_FROM_DT_TM").desc_nulls_last(),
    F.col("BINARY_SIZE").desc_nulls_last())
cand = (cand0.withColumn("_rn", F.row_number().over(w_key))
             .where(F.col("_rn") == 1)
             .select("EVENT_ID", "ADC_UPDT", "raw_sha256", "CONTENT_TYPE",
                     "BINARY_SIZE", "ENCNTR_ID", "Trust"))

# Retry-aware anti-join: skip only TERMINAL manifest keys; retryable failures re-enter.
if table_exists(MANIFEST_TABLE):
    terminal = (spark.table(MANIFEST_TABLE)
                .where(F.col("STATUS").isin(TERMINAL_STATUSES))
                .select("EVENT_ID", "raw_sha256").distinct())
    cand = cand.join(F.broadcast(terminal), ["EVENT_ID", "raw_sha256"], "left_anti")

n_cand = cand.count()  # small set; no serverless caching (see feedback_serverless_no_cache)
print(f"candidates (file keys to process): {n_cand}")

In [0]:
# ==================== PERSON_ID MAP (current-row window) ====================
# clinical_event is versioned: use current-valid rows + Trust, one PERSON_ID per EVENT_ID.
w_ce = Window.partitionBy("EVENT_ID").orderBy(F.col("VALID_FROM_DT_TM").desc_nulls_last())
pid_map = (spark.table(CLINEVENT_TABLE)
    .where(F.col("Trust") == TRUST)
    .where(F.col("VALID_UNTIL_DT_TM") > F.lit("2099-01-01").cast("timestamp"))
    .select("EVENT_ID", F.col("PERSON_ID").cast("long").alias("PERSON_ID"), "VALID_FROM_DT_TM")
    .withColumn("_rn", F.row_number().over(w_ce)).where(F.col("_rn") == 1)
    .select("EVENT_ID", "PERSON_ID"))

In [0]:
# ==================== RAW RE-READ (exact version) ====================
# Read raw chunks for the candidate versions, dedup per (EVENT_ID, ADC_UPDT, BLOB_SEQ_NUM) with
# Blob 1's window, assemble. Drive the work-set FROM `cand` (LEFT join) so a candidate whose raw
# chunks are absent (aged out / not staged) is retained and recorded as retryable "Raw missing".
cand_keys = cand.select("EVENT_ID", "ADC_UPDT").distinct()

src = (spark.table(SOURCE_TABLE).where(F.col("Trust") == TRUST)
       .join(F.broadcast(cand_keys), ["EVENT_ID", "ADC_UPDT"], "inner"))

w_chunk = Window.partitionBy("EVENT_ID", "ADC_UPDT", "BLOB_SEQ_NUM").orderBy(
    F.col("VALID_UNTIL_DT_TM").desc_nulls_last(),
    F.col("UPDT_DT_TM").desc_nulls_last(),
    F.col("LAST_UTC_TS").desc_nulls_last())
src_dedup = src.withColumn("_rn", F.row_number().over(w_chunk)).where(F.col("_rn") == 1)

grouped = (src_dedup.groupBy("EVENT_ID", "ADC_UPDT").agg(
    F.sort_array(F.collect_list(F.struct(
        F.col("BLOB_SEQ_NUM").alias("BLOB_SEQ_NUM"),
        F.col("BLOB_CONTENTS").alias("BLOB_CONTENTS")))).alias("chunks"),
    F.max("COMPRESSION_CD").alias("COMPRESSION_CD"),
    F.max("BLOB_LENGTH").alias("BLOB_LENGTH"),
    F.min("VALID_FROM_DT_TM").alias("VALID_FROM_DT_TM")))

work = (cand
    .join(grouped, ["EVENT_ID", "ADC_UPDT"], "left")   # brings chunks / COMPRESSION_CD / BLOB_LENGTH / VALID_FROM_DT_TM
    .join(pid_map, ["EVENT_ID"], "left"))

In [0]:
# ==================== DECODE + WRITE (bounded, atomic, validated) ====================
import pypdfium2 as pdfium
from PIL import Image

def _yyyymm(vfrom, adc):
    ts = vfrom or adc
    if ts is None:
        return ("0000", "00")
    return (f"{ts.year:04d}", f"{ts.month:02d}")

def _validate_file(path, ext):
    """Structural validation. Returns (ok, error_or_None)."""
    try:
        if ext == "pdf":
            d = pdfium.PdfDocument(path)
            n = len(d)
            d.close()
            if n <= 0:
                return (False, "pdf has 0 pages")
        else:  # tif / jpg / png
            with Image.open(path) as im:
                im.verify()
        return (True, None)
    except Exception as e:
        return (False, f"validate:{type(e).__name__}:{e}")

results = []  # tuples matching batch schema (extracted_ts/date added in Spark)

def _emit(EVENT_ID, PERSON_ID, ENCNTR_ID, Trust, ADC_UPDT, content_type, file_ext,
          byte_size, blob_length_expected, raw_sha256, file_sha256, volume_path,
          strategy, status, last_error):
    results.append((
        int(EVENT_ID) if EVENT_ID is not None else None,
        int(PERSON_ID) if PERSON_ID is not None else None,
        int(ENCNTR_ID) if ENCNTR_ID is not None else None,
        Trust, ADC_UPDT, content_type, file_ext,
        int(byte_size) if byte_size is not None else None,
        int(blob_length_expected) if blob_length_expected is not None else None,
        raw_sha256, file_sha256, volume_path,
        int(DECOMPRESSOR_VERSION), strategy, status, 1, last_error, RUN_ID, SOURCE))

n_written = 0
for row in work.toLocalIterator():
    eid = int(row["EVENT_ID"])
    adc = row["ADC_UPDT"]
    expected_sha = row["raw_sha256"]
    enc = row["ENCNTR_ID"]
    trust = row["Trust"]
    pid = row["PERSON_ID"]
    ct = row["CONTENT_TYPE"]
    blob_length = int(row["BLOB_LENGTH"]) if row["BLOB_LENGTH"] is not None else None
    try:
        chunk_structs = row["chunks"]
        # (0) RAW MISSING — candidate has no raw chunks in SOURCE_TABLE (aged out / not staged).
        if not chunk_structs:
            _emit(eid, pid, enc, trust, adc, ct, "unknown", None, None,
                  expected_sha, None, None, None, "Raw missing",
                  "no raw chunks for (EVENT_ID, ADC_UPDT) in SOURCE_TABLE")
            continue

        chunks = [{"BLOB_SEQ_NUM": c["BLOB_SEQ_NUM"],
                   "BLOB_CONTENTS": (bytes(c["BLOB_CONTENTS"]) if c["BLOB_CONTENTS"] else None)}
                  for c in chunk_structs]
        combined = b"".join(c["BLOB_CONTENTS"] for c in chunks if c["BLOB_CONTENTS"])

        # (1) HASH VERIFY — guarantees we reconstructed the exact bytes the run decoded.
        my_sha = raw_content_sha256(chunks)
        if my_sha != expected_sha:
            _emit(eid, pid, enc, trust, adc, ct, "unknown", None, blob_length,
                  expected_sha, None, None, None, "Raw version mismatch",
                  f"reassembled={my_sha[:12]} expected={str(expected_sha)[:12]}")
            continue

        # (2) PRE-DECODE SIZE GATE (true decompressed size)
        if blob_length is not None and blob_length > FILE_MAX_SIZE:
            _emit(eid, pid, enc, trust, adc, ct, "unknown", None, blob_length,
                  expected_sha, None, None, None, "Skipped - oversize",
                  f"blob_length={blob_length}>{FILE_MAX_SIZE}")
            continue

        # (3) DECODE
        metrics = {}
        dec, err = decompress(combined, row["COMPRESSION_CD"], len(chunks), blob_length, metrics)
        strat = metrics.get("decompression_strategy")
        if dec is None:
            _emit(eid, pid, enc, trust, adc, ct, "unknown", None, blob_length,
                  expected_sha, None, None, strat, "Decode failed", str(err))
            continue

        # (4) POST-DECODE SIZE GATE
        if len(dec) > FILE_MAX_SIZE:
            _emit(eid, pid, enc, trust, adc, ct, "unknown", len(dec), blob_length,
                  expected_sha, None, None, strat, "Skipped - oversize",
                  f"decoded={len(dec)}>{FILE_MAX_SIZE}")
            continue

        # (5) CLASSIFY BY MAGIC BYTES
        cls = classify_file_type(dec)
        if cls is None:
            _emit(eid, pid, enc, trust, adc, ct, "unknown", len(dec), blob_length,
                  expected_sha, None, None, strat, "Skipped - not in-scope type",
                  f"magic={dec[:8].hex()}")
            continue
        file_ext, subfolder, content_type = cls

        # (6) LENGTH EXACTNESS (byte-exact to BLOB_LENGTH; mismatch => truncation/mis-decode)
        if blob_length is not None and len(dec) != blob_length:
            _emit(eid, pid, enc, trust, adc, content_type, file_ext, len(dec), blob_length,
                  expected_sha, None, None, strat, "Validation failed",
                  f"len={len(dec)} != BLOB_LENGTH={blob_length}")
            continue

        # (7) ATOMIC WRITE: temp -> validate -> rename
        yyyy, mm = _yyyymm(row["VALID_FROM_DT_TM"], adc)
        out_dir = f"{TARGET_VOLUME}/{subfolder}/{yyyy}/{mm}"
        os.makedirs(out_dir, exist_ok=True)
        sha8 = my_sha[:8]
        pid_tag = f"p{pid}" if pid is not None else "pNA"
        final_name = f"{subfolder}_{pid_tag}_e{eid}_{sha8}.{file_ext}"
        tmp_path = os.path.join(out_dir, f".tmp_{sha8}.{file_ext}")
        final_path = os.path.join(out_dir, final_name)

        with open(tmp_path, "wb") as fh:
            fh.write(dec)
            fh.flush()
            os.fsync(fh.fileno())

        ok, verr = _validate_file(tmp_path, file_ext)
        if not ok:
            try:
                os.remove(tmp_path)
            except OSError:
                pass
            _emit(eid, pid, enc, trust, adc, content_type, file_ext, len(dec), blob_length,
                  expected_sha, None, None, strat, "Validation failed", verr)
            continue

        os.replace(tmp_path, final_path)
        file_sha = hashlib.sha256(dec).hexdigest()
        _emit(eid, pid, enc, trust, adc, content_type, file_ext, len(dec), blob_length,
              expected_sha, file_sha, final_path, strat, "Extracted", None)
        n_written += 1
    except Exception as e:
        _emit(eid, pid, enc, trust, adc, ct, "unknown", None, blob_length,
              expected_sha, None, None, None, "Decode failed",
              f"{type(e).__name__}:{e}\n{traceback.format_exc()[:500]}")

print(f"processed {len(results)} file keys; {n_written} files written")

In [0]:
# ==================== MANIFEST UPSERT (retry-aware) ====================
if results:
    batch_schema = ("EVENT_ID long, PERSON_ID long, ENCNTR_ID long, Trust string, ADC_UPDT timestamp, "
                    "content_type string, file_ext string, byte_size long, blob_length_expected long, "
                    "raw_sha256 string, file_sha256 string, volume_path string, decompressor_version int, "
                    "decompression_strategy string, STATUS string, attempt_count int, last_error string, "
                    "run_id string, source string")
    batch = spark.createDataFrame(results, batch_schema)

    # Dedup MERGE source to one row per file key (defensive; prefer Extracted, then larger).
    w_b = Window.partitionBy("EVENT_ID", "raw_sha256").orderBy(
        F.when(F.col("STATUS") == "Extracted", 0).otherwise(1),
        F.col("byte_size").desc_nulls_last())
    batch = (batch.withColumn("_rn", F.row_number().over(w_b)).where(F.col("_rn") == 1).drop("_rn")
                  .withColumn("extracted_ts", F.current_timestamp())
                  .withColumn("extracted_date", F.current_date()))

    batch.createOrReplaceTempView("_blob4_batch")
    spark.sql(f"""
        MERGE INTO {MANIFEST_TABLE} t
        USING _blob4_batch s
        ON t.EVENT_ID = s.EVENT_ID AND t.raw_sha256 = s.raw_sha256
        WHEN MATCHED THEN UPDATE SET
            t.PERSON_ID = s.PERSON_ID, t.ENCNTR_ID = s.ENCNTR_ID, t.Trust = s.Trust,
            t.ADC_UPDT = s.ADC_UPDT, t.content_type = s.content_type, t.file_ext = s.file_ext,
            t.byte_size = s.byte_size, t.blob_length_expected = s.blob_length_expected,
            t.file_sha256 = s.file_sha256, t.volume_path = s.volume_path,
            t.decompressor_version = s.decompressor_version,
            t.decompression_strategy = s.decompression_strategy,
            t.STATUS = s.STATUS, t.attempt_count = COALESCE(t.attempt_count, 0) + 1,
            t.last_error = s.last_error, t.run_id = s.run_id, t.source = s.source,
            t.extracted_ts = s.extracted_ts, t.extracted_date = s.extracted_date
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"merged {batch.count()} rows into {MANIFEST_TABLE}")
else:
    print("no candidates to merge")

In [0]:
# ==================== SUMMARY ====================
if results:
    (spark.table(MANIFEST_TABLE)
        .where(F.col("run_id") == F.lit(RUN_ID))
        .groupBy("STATUS", "file_ext").count().orderBy(F.desc("count"))
        .show(50, False))
print("BLOB4_DONE")